# 200 parches y un mes

Somos el equipo de seguridad de una empresa. Nos llegan **200 vulnerabilidades** para parchear este mes y no damos abasto.

Tenemos un catálogo con casi **90.000 vulnerabilidades reales**.

## Con qué vamos a trabajar: el catálogo CVE

**CVE** (Common Vulnerabilities and Exposures) es el catálogo público mundial de vulnerabilidades de software. Cada fallo que se descubre recibe un código único, tipo `CVE-2014-0160`: el año en que se registró y un número. Lo mantiene MITRE, y el NIST, un organismo público de Estados Unidos, le añade la nota de gravedad en su base de datos NVD, de donde sale esta copia. Lo consulta cualquier equipo de seguridad del mundo.

Vulnerabilidades registradas **entre 1999 y noviembre de 2019**:

- **89.660 filas**, una por vulnerabilidad
- **13 columnas**. Las que vamos a usar:

| Columna | Qué es |
|---|---|
| `cve_id` | el código único de la vulnerabilidad. En el archivo viene con un nombre raro, `Unnamed: 0`; lo arreglaremos |
| `pub_date` | cuándo se publicó |
| `cvss` | la **nota de gravedad**, de 0 a 10, calculada con un estándar que se llama CVSS |
| `cwe_code` y `cwe_name` | el **tipo de fallo**. Un CVE es un fallo concreto en un producto concreto; el CWE es la categoría de error a la que pertenece: inyección SQL, XSS, error de memoria... `cwe_code` es el número de esa categoría y `cwe_name` su nombre. Hay 170 categorías en este catálogo |

Las otras columnas describen el fallo con más detalle (desde dónde se ataca, qué daño hace) y no las vamos a necesitar: con las de la tabla se contestan las tres preguntas.

El número de `cwe_code` no mide nada: es un identificador fijo, como un código postal. El 89 es siempre inyección SQL en todo el mundo, y no es peor que el 79 por ser mayor. Lo grave que es un fallo lo dice `cvss`; el tipo de fallo lo dice `cwe_code`. Son dos cosas distintas.

Una fila de ejemplo, **Heartbleed**, el fallo de OpenSSL que salió en todos los telediarios en 2014:

| Columna | Valor |
|---|---|
| `cve_id` | CVE-2014-0160 |
| `pub_date` | 7 de abril de 2014 |
| `cvss` | 5,0 |
| `cwe_code` | 119 |
| `cwe_name` | Improper Restriction of Operations within the Bounds of a Memory Buffer |
| `summary` | "The TLS and DTLS implementations in OpenSSL 1.0.1 before 1.0.1g do not properly handle Heartbeat Extension packets..." |

## Qué es un EDA

Un **EDA** (Análisis Exploratorio de Datos) es el proceso inicial para comprender un conjunto de datos. Consiste en mirarlos antes de usarlos: ver qué contienen, detectar valores que faltan, anomalías y valores atípicos, y entender las características de lo que tenemos entre manos.

**Objetivo:** comprender y explorar los datos antes de sacar conclusiones o construir modelos con ellos.

## Lo que queremos saber

1. ¿Cuántas son graves de verdad?
2. ¿Hay tipos de fallo peores que otros? Por ejemplo, ¿SQL Injection o XSS?
3. ¿Qué tipos de fallo deberíamos revisar primero?

## 🙋 ¿Qué deberíamos hacer para saberlo?

Pistas:
- ¿Sabemos qué hay en la tabla?
- ¿Nos podemos fiar de todos los datos?
- ¿Qué columna nos dice lo grave que es una vulnerabilidad?

## El plan

1. Miramos qué hay.
2. Limpiamos.
3. Medimos una variable.
4. Cruzamos dos.
5. Sacamos conclusiones.

Mirar qué hay viene hecho, y contar los huecos también. Desde limpiar, decides tú.

## Las 10 fases de un EDA

El plan de arriba es el resumen. Un EDA completo pasa por estas fases:

| | Fase | Dónde |
|---|---|---|
| 1 | Definición del problema | el caso y las tres preguntas |
| 2 | Recopilación de datos | bloques 0 y 1 |
| 3 | Limpieza de datos | bloques 2 y 3 |
| 4 | Análisis univariado: una variable sola | bloque 4 |
| 5 | Análisis bivariado: dos variables cruzadas | bloques 5 y 6 |
| 6 | Análisis multivariado: tres o más a la vez | no entra en este notebook |
| 7 | Visualización de datos | un gráfico en cada bloque |
| 8 | Identificación de patrones | lo que sale en cada bloque |
| 9 | Formulación de hipótesis | lo que apuestas antes de ejecutar cada celda |
| 10 | Conclusiones | cierre |

El multivariado no entra: con cruzar dos variables llega para contestar las tres preguntas.

## Bloque 0 · Cargar librerías

Ya viene hecho. Solo tienes que ejecutarlo.

In [ ]:
import warnings
warnings.filterwarnings('ignore', module='tqdm')

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import kagglehub
import os

sns.set_style("whitegrid")
plt.rcParams['figure.figsize'] = (10, 6)

path = kagglehub.dataset_download("andrewkronser/cve-common-vulnerabilities-and-exposures/versions/3")
print("Listo")

## Bloque 1 · ¿Qué hay en la tabla?

Lo primero siempre: abrir, ver el tamaño y mirar las primeras filas.

Las columnas que nos importan: **`cvss`** es la nota de gravedad, de 0 a 10. **`cwe_code`** es el número del tipo de fallo (89 es inyección SQL, 79 es XSS) y **`cwe_name`** su nombre largo.

In [ ]:
df = pd.read_csv(os.path.join(path, 'cve.csv'))

print(f"Filas: {df.shape[0]}, columnas: {df.shape[1]}")
df.head(3)

**Qué ha salido**

89.660 filas y 13 columnas. `Unnamed: 0` es el identificador mal guardado. 

## Bloque 2 · ¿Podemos fiarnos de los datos?

Hay celdas vacías (`NaN`). Contamos cuántas hay en cada columna, y en cuántas filas.

In [ ]:
print(df.isnull().sum())

print("\nFilas con algún hueco:", df.isnull().any(axis=1).sum())

**Qué ha salido**

Seis columnas con 884 huecos cada una, y 884 filas con algún hueco: faltan las seis a la vez en las mismas filas. Un 1% del total.

---

# A partir de aquí

El código tiene huecos marcados con `____`. En cada hueco tienes que tomar una decisión: qué columna, dónde cortar, qué función. Encima de cada celda están las preguntas que llevan a cada hueco. Decide, rellena y ejecuta.

Debajo de cada bloque hay una celda con lo que se decidió y lo que tiene que salir. Mírala después de ejecutar, no antes.

## Bloque 3 · Limpiar

🙋 **Hay 884 filas con seis datos vacíos. ¿Las rellenamos con algún valor, o las borramos? ¿Por qué?**

🙋 **¿Alguna columna con nombre raro que convenga arreglar?**

Opciones: `dropna()` o `fillna(...)`

In [ ]:
# ¿Borrar las filas incompletas o rellenarlas?
df = df.____()

# Arreglamos el nombre raro por cve_id
df = df.rename(columns={'Unnamed: 0': '____'})

print(f"Nos quedamos con {len(df)} vulnerabilidades")

In [ ]:
# Prueba: copia la celda de arriba, cambia solo un hueco y compara


**Lo que decidimos**

- Borrarlas. A esas 884 filas les faltan seis columnas a la vez y son un 1% del total: rellenarlas sería inventarse seis datos por fila. Si faltara una sola columna en muchas filas, rellenar tendría más sentido, porque borrar tiraría mucha información buena por un solo dato.
- `Unnamed: 0` pasa a llamarse `cve_id`.

**Qué ha salido**

88.776 vulnerabilidades. Si te sale otro número, revisa el primer hueco.

## Bloque 4 · Pregunta 1 · ¿Cuántas son graves de verdad?

🙋 **¿Qué columna nos dice lo grave que es una vulnerabilidad?**

🙋 **Va del 0 al 10. ¿Cómo decidimos cuáles llamamos "graves"?**

🙋 **¿Cómo contamos cuántas hay de cada grupo, y cómo lo dibujamos?**

Opciones: para el gráfico, `'bar'` o `'line'`

In [ ]:
# Partimos la nota en grupos. ¿Qué columna? ¿Dónde cortamos?
df['gravedad'] = pd.cut(df['____'],
                        bins=[0, ____, ____, 10],
                        labels=['Baja', 'Media', 'Alta'],
                        include_lowest=True)

# Contamos cuántas hay de cada grupo
conteo = df['gravedad'].value_counts().sort_index()
print(conteo)

# ¿Qué gráfico?
conteo.plot(kind='____', color=['green', 'orange', 'red'])
plt.title('Vulnerabilidades por gravedad')
plt.xticks(rotation=0)
plt.show()

In [ ]:
# Prueba: copia la celda de arriba, cambia solo un hueco y compara


**Lo que decidimos**

- `cvss`.
- Partiendo la nota en grupos. Usamos los cortes oficiales: Baja hasta 3,9, Media hasta 6,9, Alta de 7 a 10.
- Contando cada grupo y con un gráfico de barras.

**Qué ha salido**

7.695 bajas, 51.636 medias y 29.445 altas. **Una de cada tres es grave.**

## Bloque 5 · Pregunta 2 · ¿Hay tipos de fallo peores que otros?

🙋 **¿Tú qué apuestas: SQL Injection o XSS? Decídelo antes de ejecutar.**

🙋 **¿Qué columna nos dice el tipo de fallo? ¿Nos sirve tal cual?**

🙋 **¿Cómo comparamos la nota de un tipo con la de otro?**

🙋 **¿Qué gráfico lo enseña de un vistazo?**

Opciones: `.mean()` o `.median()` · `sns.boxplot` o `sns.barplot`

| Código | Tipo | Qué es |
|---|---|---|
| 89 | SQL Injection | comandos de base de datos metidos en un formulario |
| 77, 78 | Command Injection | comandos del sistema metidos donde no toca |
| 94 | Code Injection | hacer que la aplicación ejecute código del atacante |
| 119 | Errores de memoria | escribir o leer fuera de lo que toca en la memoria |
| 352 | CSRF | engañar al navegador de la víctima para que haga algo |
| 22 | Path Traversal | rutas tipo `../../` para leer archivos prohibidos |
| 79 | XSS | código que se ejecuta en el navegador de otros usuarios |

**Mediana:** la nota del medio si pones todas en fila. **Boxplot:** la caja es la mitad de los valores del medio, la raya es la mediana y los circulitos son casos raros.

In [ ]:
# Los códigos de la tabla, con nombre corto
tipos = {89: 'SQL Injection', 77: 'Command Injection', 78: 'Command Injection',
         94: 'Code Injection', 119: 'Errores de memoria', 352: 'CSRF',
         22: 'Path Traversal', 79: 'XSS'}

# Nos quedamos solo con esos tipos y les ponemos el nombre
subset = df[df['cwe_code'].isin(tipos.keys())].copy()
subset['tipo'] = subset['cwe_code'].map(tipos)
print("Cuántas hay de cada tipo:")
print(subset['tipo'].value_counts())

# ¿Agrupamos por qué columna? ¿Qué nota comparamos? ¿Media o mediana?
medianas = subset.groupby('____')['____'].____().sort_values(ascending=False)
print("\nNota mediana de cada tipo:")
print(medianas)

# ¿Qué gráfico?
sns.____(data=subset, x='tipo', y='cvss', order=medianas.index, color='lightgreen')
plt.title('Gravedad por tipo de vulnerabilidad')
plt.xticks(rotation=20)
plt.show()

In [ ]:
# Prueba: copia la celda de arriba, cambia solo un hueco y compara


**Lo que decidimos**

- Gana SQL Injection.
- `cwe_name` tiene el tipo, pero los nombres son de 80 letras y no caben en un gráfico. Usamos `cwe_code` y un diccionario con nombres cortos: los códigos están en la tabla.
- Agrupando por tipo y mirando la mediana de cada uno. Si has puesto la media, también gana SQL Injection a XSS; la mediana es mejor porque no se deja arrastrar por casos raros.
- Un boxplot. Si has puesto barras, también vale.

**Qué ha salido**

SQL Injection 7,5 contra XSS 4,3. **La SQL Injection es mucho más grave**: va directa a la base de datos; el XSS se ejecuta en el navegador de la víctima y necesita que alguien pique. Arriba del todo sale Command Injection con 9, que también es una inyección. Y hay más del doble de XSS (12.242) que de SQL Injection (5.742): cantidad y gravedad son cosas distintas. Si SQL Injection o CSRF salen como una raya, no está roto: tres de cada cuatro valen exactamente lo mismo.

## Bloque 6 · Pregunta 3 · ¿Qué tipos revisamos primero?

🙋 **¿Qué dos columnas tenemos que cruzar?**

🙋 **¿Contamos cuántas graves hay en cada tipo, o miramos el porcentaje de graves? ¿Qué nos dice cada una?**

🙋 **¿Cómo lo ordenamos para que el gráfico se lea solo?**

Opciones: `normalize='index'` da porcentajes por fila · `ascending=True` o `False`

In [ ]:
# Ponemos el tipo a toda la tabla; lo que no está en la lista va a "Otras"
df['tipo'] = df['cwe_code'].map(tipos).fillna('Otras')

# ¿Qué dos columnas cruzamos?
cruce = pd.crosstab(df['____'], df['____'])
print("Cuántas graves hay en cada tipo:")
print(cruce['Alta'].sort_values(ascending=False))

# ¿Contamos o porcentaje?
porcentaje = pd.crosstab(df['tipo'], df['gravedad'], normalize='____') * 100
# ¿Cómo lo ordenamos?
porcentaje = porcentaje.sort_values('____', ascending=____)
print("\nPorcentaje de graves en cada tipo:")
print(porcentaje['Alta'].round(1))

porcentaje.plot(kind='bar', stacked=True, color=['green', 'orange', 'red'])
plt.title('Porcentaje de graves por tipo')
plt.legend(title='Gravedad', bbox_to_anchor=(1, 1), loc='upper left')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
# Prueba: copia la celda de arriba, cambia solo un hueco y compara


**Lo que decidimos**

- El tipo de fallo y la gravedad que creamos en el bloque 4.
- Las dos valen y dicen cosas distintas. Contar dice dónde hay más trabajo grave. El porcentaje dice qué tipo es peligroso por naturaleza. Miramos las dos.
- De más a menos porcentaje de graves.

**Qué ha salido**

Por porcentaje: SQL Injection 77,7%, Command Injection 76,7%, Code Injection 64,2%, Errores de memoria 62,3%... y XSS 0,1%. **Las inyecciones son lo más peligroso por naturaleza.** Por cantidad: donde más graves hay es en Errores de memoria (7.669) y en el cajón "Otras" (13.907), que junta todos los tipos que no hemos nombrado. Las dos cosas son verdad.

## Extra · ¿Cada año hay más?

🙋 **¿Cómo contamos cuántas se publican cada año?**

Opciones: `.dt.year` o `.dt.month`

In [ ]:
# Sacamos el año de la fecha de publicación
df['year'] = pd.to_datetime(df['____']).dt.____

por_year = df[df['year'] >= 2005]['year'].value_counts().sort_index()
print(por_year)

por_year.plot(kind='line', marker='o', color='darkred')
plt.title('Vulnerabilidades publicadas por año')
plt.xticks(range(2005, 2020, 2))
plt.show()

In [ ]:
# Prueba: copia la celda de arriba, cambia solo un hueco y compara


**Lo que decidimos**

- Sacando el año de la fecha de publicación y contando cuántas hay de cada año. Empezamos en 2005 porque antes hay muy pocas y ensucian el gráfico.

**Qué ha salido**

Cada vez más: de 356 en 2005 a 14.855 en 2018. 2019 está incompleto porque el catálogo acaba en noviembre.

## Ahora tú · ¿Las vulnerabilidades de los últimos años son más graves que las de antes?

 Hacen falta dos columnas, `year` y `gravedad`, y todo lo que necesitas ya lo has usado en el bloque 6 y en el Extra.

In [ ]:
# Resuélvelo aquí


## Volvemos a las tres preguntas

1. ¿Cuántas son graves de verdad? **Una de cada tres.**
2. ¿Hay tipos de fallo peores que otros? **Sí. SQL Injection 7,5 contra XSS 4,3.**
3. ¿Qué tipos revisamos primero? **Las inyecciones, que son lo más peligroso por naturaleza. Y los errores de memoria, que es donde más graves se acumulan.**

Y las 200 de este mes: una de cada tres será grave, unas 65. Si ya traen nota, se ordenan por nota. Si no, empezamos mirando las inyecciones.

Lo que has hecho, en orden: mirar qué hay, limpiar, medir una variable, cruzar dos y sacar conclusiones. Ese orden vale para cualquier tabla.